In [0]:
# ============================================================
# CELL 1 - Configuration
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# Source configuration
# ------------------------------------------------------------

SOURCE_CATALOG = "catalog"
SOURCE_SCHEMA = "source_schema"

# Source tables to be processed
SOURCE_TABLES = [
    "customers",
    "orders",
    "order_items",
    "products"
]

# Incremental / watermark column in source tables
WATERMARK_COLUMN = "updated_At"

# Default watermark for first-time ingestion
DEFAULT_WATERMARK = "1970-01-01 00:00:00"

# ------------------------------------------------------------
# Raw layer configuration
# ------------------------------------------------------------

RAW_BASE_PATH = "gs://source"

# ------------------------------------------------------------
# Watermark control table
# ------------------------------------------------------------

WATERMARK_TABLE = "watermark_control"


# ------------------------------------------------------------
# Print configuration
# ------------------------------------------------------------

print("============================================================")
print("Source Catalog  :", SOURCE_CATALOG)
print("Source Schema   :", SOURCE_SCHEMA)
print("Source Tables   :", SOURCE_TABLES)
print("Watermark Column:", WATERMARK_COLUMN)
print("Raw Location    :", RAW_BASE_PATH)
print("Watermark Table :", WATERMARK_TABLE)
print("============================================================")

# ------------------------------------------------------------
# Display current watermark information
# ------------------------------------------------------------

print("\nCurrent Watermark Control Table:")

display(
    spark.sql(f"""
        SELECT
            *
        FROM {WATERMARK_TABLE}
        WHERE table_name IN (
            'customers',
            'orders',
            'order_items',
            'products'
        )
        ORDER BY table_name
    """)
)

In [0]:
def get_lwv(table_name):

    result = spark.sql(f"""
        SELECT
            table_name,
            status,
            lwv
        FROM {WATERMARK_TABLE}
        WHERE table_name = '{table_name}'
    """).collect()

    if not result:

        print(
            f"[WATERMARK] {table_name} not found. "
            f"Initializing with default LWV."
        )

        spark.sql(f"""
            INSERT INTO {WATERMARK_TABLE}
            VALUES (
                '{table_name}',
                'SUCCESS',
                TIMESTAMP('{DEFAULT_WATERMARK}')
            )
        """)

        print(
            f"[WATERMARK] Added {table_name} "
            f"with LWV = {DEFAULT_WATERMARK}"
        )

        return DEFAULT_WATERMARK

    current_lwv = result[0]["lwv"]

    print(
        f"[WATERMARK] {table_name} already exists. "
        f"LWV = {current_lwv}"
    )

    return current_lwv

In [0]:
# ============================================================
# CELL 3 - Read Incremental Data
# ============================================================

def read_incremental_data(table_name, lwv):
    """
    Reads records from Cloud SQL where updated_At
    is greater than the current LWV.
    """

    source_table = (
        f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{table_name}"
    )

    print(f"[SOURCE] Table : {source_table}")
    print(f"[SOURCE] LWV   : {lwv}")

    df = spark.sql(f"""
        SELECT *
        FROM {source_table}
        WHERE {WATERMARK_COLUMN} > TIMESTAMP('{lwv}')
    """)

    return df

In [0]:
# ============================================================
# CELL 4 - Calculate New LWV
# ============================================================

def get_new_lwv(df):
    """
    Gets the maximum updated_At from the incremental batch.
    """

    result = (
        df.select(
            F.max(
                F.col(WATERMARK_COLUMN)
            ).alias("new_lwv")
        )
        .collect()[0]
    )

    return result["new_lwv"]

In [0]:
# ============================================================
# CELL 5 - Write to Raw Delta
# ============================================================

def write_to_raw(df, table_name):
    """
    Appends incremental data to the Raw Delta location.
    """

    raw_path = f"{RAW_BASE_PATH}/{table_name}/"

    print(f"[RAW] Writing to: {raw_path}")

    (
        df.write
        .format("delta")
        .mode("append")
        .save(raw_path)
    )

    print(f"[RAW] Write successful: {table_name}")

In [0]:

# ============================================================
# CELL 6 - Process One Table
# ============================================================

def update_lwv(table_name, new_lwv):

    lwv_sql = new_lwv.strftime("%Y-%m-%d %H:%M:%S.%f")

    spark.sql(f"""
        UPDATE {WATERMARK_TABLE}
        SET
            lwv = TIMESTAMP('{lwv_sql}'),
            status = 'SUCCESS'
        WHERE table_name = '{table_name}'
    """)

    print(
        f"[WATERMARK] Updated {table_name} "
        f"-> LWV = {lwv_sql}"
    )


def process_table(table_name):
    """
    Executes Source -> Raw incremental processing.

    Flow:
        1. Get current LWV
        2. Read incremental source data
        3. Check for new records
        4. Calculate new LWV
        5. Write data to Raw Delta
        6. Update watermark_control

    Important:
        LWV is updated ONLY after the Raw write succeeds.
        If Raw write fails, LWV remains unchanged.
    """

    print("=" * 70)
    print(f"STARTING: {table_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # 1. Get current LWV
    # --------------------------------------------------------
    lwv = get_lwv(table_name)

    print(f"[WATERMARK] Current LWV: {lwv}")

    # --------------------------------------------------------
    # 2. Read incremental data from source
    # --------------------------------------------------------
    df_incremental = read_incremental_data(
        table_name,
        lwv
    )

    # --------------------------------------------------------
    # 3. Check whether new records exist
    # --------------------------------------------------------
    record_count = df_incremental.count()

    print(f"[SOURCE] Record count: {record_count}")

    if record_count == 0:
        print(f"[INFO] No new data for {table_name}")
        return None

    # --------------------------------------------------------
    # 4. Calculate new LWV
    # --------------------------------------------------------
    new_lwv = get_new_lwv(df_incremental)

    print(f"[WATERMARK] New LWV: {new_lwv}")

    # --------------------------------------------------------
    # 5. Write incremental data to Raw Delta
    # --------------------------------------------------------
    # If this fails, the code stops here and
    # watermark_control is NOT updated.
    write_to_raw(
        df_incremental,
        table_name
    )

    print(f"[RAW] Write successful: {table_name}")

    # --------------------------------------------------------
    # 6. Update watermark ONLY after Raw write succeeds
    # --------------------------------------------------------
    update_lwv(
        table_name,
        new_lwv
    )

    print(f"[SUCCESS] Completed: {table_name}")
    print("=" * 70)

    return new_lwv




In [0]:
# ============================================================
# CELL 7 - Run All 4 Tables
# ============================================================

results = {}

for table_name in SOURCE_TABLES:

    print("\n")
    print("#" * 70)
    print(f"PROCESSING TABLE: {table_name}")
    print("#" * 70)

    new_lwv = process_table(table_name)

    results[table_name] = new_lwv

print("\n")
print("=" * 70)
print("ALL TABLES COMPLETED")
print("=" * 70)

for table_name, new_lwv in results.items():
    print(f"{table_name:15} -> {new_lwv}")

In [0]:
# ============================================================
# CELL 8 - Write Ingestion Log to GCS
# ============================================================

from datetime import datetime

run_time = datetime.now()

log_lines = []

log_lines.append("=" * 60)
log_lines.append("SOURCE TO RAW INGESTION LOG")
log_lines.append("=" * 60)
log_lines.append(f"Run Time: {run_time}")
log_lines.append("")

for table_name, new_lwv in results.items():

    log_lines.append(f"Table       : {table_name}")

    if new_lwv is not None:
        log_lines.append("Status      : SUCCESS")
        log_lines.append(f"New LWV     : {new_lwv}")
    else:
        log_lines.append("Status      : NO NEW DATA")
        log_lines.append("New LWV     : No change")

    log_lines.append("")


log_lines.append("=" * 60)
log_lines.append("ALL TABLES PROCESSED")
log_lines.append("=" * 60)

log_content = "\n".join(log_lines)

# Create unique log filename
log_timestamp = run_time.strftime("%Y%m%d_%H%M%S")

log_path = (
    f"{RAW_BASE_PATH}/logs/"
    f"ingestion_{log_timestamp}.txt"
)

# Write text file to GCS
dbutils.fs.put(
    log_path,
    log_content,
    overwrite=True
)

print(f"[LOG] Written successfully:")
print(log_path)